# Tutorial 51: Manual Creation of Element Dataframes

This Example demonstrates the capabilities of the class Dataframes_SIR3S_Model that extends SIR3S_Model be abilities to work directley with pandas dataframes. It is shown how to create dataframes containing information about elements such as Nodes, Pipes, etc. existing in a SIR 3S Model. The methods presented are manual, user-defined and detailed. For creating more general dataframes with less input necessary, see Tutorial 52.   

# Toolkit Release

In [1]:
#pip install 

# Imports

## SIR 3S Toolkit

### Regular Import/Init

In [2]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [3]:
from sir3stoolkit.core import wrapper

In [4]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [5]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

[2026-06-08 10:25:10,820] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-06-08 10:25:10,820] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-06-08 10:25:10,882] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


### Additional Import/Init for Dataframes class

In [6]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [7]:
s3s = SIR3S_Model_Dataframes()

[2026-06-08 10:25:23,888] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


## Additional

In [8]:
import pandas as pd
from shapely.geometry import Point
import re
import folium
from folium.plugins import HeatMap
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import contextily as cx

# Open Model

In [9]:
s3s.OpenModel(dbName=r"Toolkit_Tutorial51_Model.db3",
              providerType=s3s.ProviderTypes.SQLite,
              Mid="M-1-0-1",
              saveCurrentlyOpenModel=False,
              namedInstance="",
              userID="",
              password="")

[2026-06-08 10:25:39,238] INFO in sir3stoolkit.core.wrapper: Model is open for further operation


# Calculate Model

In [10]:
#s3s.ExecCalculation(True) # To ensure result data

# Manual Dataframe creation

## model_data - Nodes

We can use the [generate_element_model_data_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.Dataframes_SIR3S_Model.generate_element_model_data_dataframe) method to obtain a dataframe for all instances of a specifc component type (Nodes, Pipes, etc.) with user defined model_data properties (all non-result data related to a component) to be added. For this example we will use nodes.

In this case we are interested in Nodes, let's check what model_data properties are available for this element.

In [11]:
s3s.GetPropertiesofElementType(s3s.ObjectTypes.Node)

['Name',
 'Ktyp',
 'Zkor',
 'QmEin',
 'Lfakt',
 'Fkpzon',
 'Fkfstf',
 'Fkutmp',
 'Fkfqps',
 'Fkcont',
 'Fk2lknot',
 'Beschreibung',
 'Idreferenz',
 'Iplanung',
 'Kvr',
 'Qakt',
 'Xkor',
 'Ykor',
 'NodeNamePosition',
 'ShowNodeName',
 'KvrKlartext',
 'NumberOfVERB',
 'HasBlockConnection',
 'Tk',
 'Pk',
 'InVariant',
 'GeometriesDiffer',
 'SymbolFactor',
 'bz.Drakonz',
 'bz.Fk',
 'bz.Fkpvar',
 'bz.Fkqvar',
 'bz.Fklfkt',
 'bz.PhEin',
 'bz.Tm',
 'bz.Te',
 'bz.PhMin']

In the below subcases differnt choices of parameters for creating the model_data dataframe manually are shown. Note that not all possible combination of paramters are shown here.

### Subcase 1: Minimal

Let's say for now we only want a dataframe with all node tks without any related data.

In [12]:
(s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Node
                                        ,properties=[]
                                        )).head(3)

[2026-06-08 10:25:39,309] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node
[2026-06-08 10:25:39,317] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 517 element(s) of element type ObjectTypes.Node.
[2026-06-08 10:25:39,348] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 0 model_data properties.
[2026-06-08 10:25:39,350] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving ...
[2026-06-08 10:25:39,356] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (517, 1)


,tk
0,5669301360686511351
1,5397948523091900401
2,5239335112004772156


### Subcase 2: Specific Properties 

Let's say we now want the minimal dataframe extended with supply/return info (kvr) and height (Zkor) of the nodes.

In [13]:
(s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Node
                                        ,properties=["Kvr", "Zkor"]
                                        )).head(3)

[2026-06-08 10:25:39,398] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node
[2026-06-08 10:25:39,403] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 517 element(s) of element type ObjectTypes.Node.
[2026-06-08 10:25:39,409] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 2 model_data properties.
[2026-06-08 10:25:39,411] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Kvr', 'Zkor']...
[2026-06-08 10:25:39,517] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (517, 3)


,tk,Kvr,Zkor
0,5669301360686511351,1,554.04
1,5397948523091900401,1,556.16
2,5239335112004772156,1,561.01


### Subcase 3: Maximal

Let's say now we want basically all available model_data (non-result data) related to nodes. 

If geometry=True, the 2D Point geometry will be added for Nodes and it will be attempted to transform the DataFrame into a GeoDataFrame using the SRID (eg. EPSG: 25832) defined in the model. If no SRID is defined, the DataFrame will remain as a DataFrame, but the geometry column with be added nonetheless.

In [14]:
(s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Node
                                        ,properties=None # Instead of an empty or filled list, we pass None to indicate we want all available properties
                                        ,geometry=True # geometry data has to be explicitly requested
                                        )).head(3)

[2026-06-08 10:25:39,555] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node
[2026-06-08 10:25:39,561] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 517 element(s) of element type ObjectTypes.Node.
[2026-06-08 10:25:39,562] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.Node.
[2026-06-08 10:25:39,562] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 37 model_data properties.
[2026-06-08 10:25:39,562] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Name', 'Ktyp', 'Zkor', 'QmEin', 'Lfakt', 'Fkpzon', 'Fkfstf', 'Fkutmp', 'Fkfqps', 'Fkcont', 'Fk2lknot', 'Beschreibung', 'Idreferenz', 'Iplanung', 'Kvr', 'Qakt', 'Xkor', 'Ykor', 'NodeNamePosition', 'ShowNodeName', 'KvrKlartext', 'NumberOfVERB', 'HasBlockConnection', 'Tk', 'Pk', 'InVariant', 'Geo

,tk,Name,Ktyp,Zkor,QmEin,Lfakt,Fkpzon,Fkfstf,Fkutmp,Fkfqps,Fkcont,Fk2lknot,Beschreibung,Idreferenz,Iplanung,Kvr,Qakt,Xkor,Ykor,NodeNamePosition,ShowNodeName,KvrKlartext,NumberOfVERB,HasBlockConnection,Tk,Pk,InVariant,GeometriesDiffer,SymbolFactor,bz.Drakonz,bz.Fk,bz.Fkpvar,bz.Fkqvar,bz.Fklfkt,bz.PhEin,bz.Tm,bz.Te,bz.PhMin,geometry
0,5669301360686511351,V-K03S,QKON,554.04,0,1,5520728169779652386,4798673252636751115,5591325053703727727,-1,5029128874972463118,5761544923588724980,Abzw. Am Sonnenbühl,3S3768E2953006F125B34916F6A1681667,1,1,0,713620.267807,5.578828e+06,1,False,Vorlauf,0,False,5669301360686511351,5669301360686511351,False,False,0.2,0,5669301360686511351,-1,-1,-1,0,0,0,0,POINT (713620.268 5578828.419)
1,5397948523091900401,V-K13S,QKON,556.16,0,1,5520728169779652386,4798673252636751115,5591325053703727727,-1,5029128874972463118,4749864932928780363,Anfangsknoten generiert von SirDB,3SC30ED954AA2F1D59B1F92FFEBCCB7B5B,1,1,0,713602.294600,5.578860e+06,1,False,Vorlauf,0,False,5397948523091900401,5397948523091900401,False,False,0.2,0,5397948523091900401,-1,-1,-1,0,0,0,0,POINT (713602.295 5578860.106)
2,5239335112004772156,V-K23S,QKON,561.01,0,1,5520728169779652386,4798673252636751115,5591325053703727727,-1,5029128874972463118,4899862460906451371,Anfangsknoten generiert von SirDB,3SC30ED954AA2F1D59B1F92FFEBCCB7B5B,1,1,0,713574.061627,5.578910e+06,1,False,Vorlauf,0,False,5239335112004772156,5239335112004772156,False,False,0.2,0,5239335112004772156,-1,-1,-1,0,0,0,0,POINT (713574.062 5578909.873)


### Subcase 4: Filtering + Element Type Col

Let's say we only want nodes inside a specific container and we also want to have an element type col adding "Node" to each row (useful when joining with other dataframes later).

In [15]:
for container_id in (s3s.GetTksofElementType(s3s.ObjectTypes.ObjectContainerSymbol)):
    print(f"{container_id}: {s3s.GetValue(container_id, 'Name')[0]}")

5029128874972463118: M-1-0-1
5027846505677995694: Erzeugung
5172832702839270493: TS
5152808213068069018: Laststeuerung
5467315850619588583: Sekundärwerte
5320319133990336755: Netztrennung


In [16]:
all_tks = s3s.GetTksofElementType(s3s.ObjectTypes.Node)

In [17]:
good_tks = []

In [ ]:
for tk in all_tks:
    if s3s.GetValue(tk, "FkCont")[0] == "5029128874972463118": # Main container
        if s3s.GetValue(tk, "Kvr")[0] == "1": # Supply/Vorlauf
            good_tks.append(tk)

In [19]:
(s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Node
                                        ,properties=[]
                                        ,tks=good_tks
                                        ,geometry=True
                                        ,element_type_col=True
                                        )).head(3)

[2026-06-08 10:25:42,312] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Node
[2026-06-08 10:25:42,318] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 517 element(s) of element type ObjectTypes.Node.
[2026-06-08 10:25:42,321] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 0 model_data properties.
[2026-06-08 10:25:42,323] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving geometry...
[2026-06-08 10:25:42,378] INFO in sir3stoolkit.mantle.dataframes: [model_data] Transforming DataFrame to GeoDataFrame successful with EPSG: 25832
[2026-06-08 10:25:42,388] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (517, 3)


,tk,geometry,element type
0,5669301360686511351,POINT (713620.268 5578828.419),Node
1,5397948523091900401,POINT (713602.295 5578860.106),Node
2,5239335112004772156,POINT (713574.062 5578909.873),Node


## Result Data - Pipes

We can use the [generate_element_model_data_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.Dataframes_SIR3S_Model.generate_element_results_dataframe) method to obtain a dataframe for all instances of a specifc component type (Nodes, Pipes, etc.) with user defined result values for certain timestamps to be added. For this example we will use pipes.

Let's check what result properties are available for Pipes.

In [20]:
s3s.GetResultProperties_from_elementType(elementType=s3s.ObjectTypes.Pipe
                                         ,onlySelectedVectors=False 
                                         )

['A',
 'ACALC',
 'CPI',
 'CPK',
 'DH',
 'DP',
 'DRAGRED',
 'DRAKONZ',
 'DSI',
 'DSK',
 'DTTR',
 'DWVERL',
 'DWVERLABS',
 'ETAAV',
 'FS',
 'HR',
 'HVEC',
 'IAKTIV',
 'IRTRENN',
 'JV',
 'JV2',
 'LAMBDA',
 'LECKEINAUS',
 'LECKMENGE',
 'LECKORT',
 'LINEPACK',
 'LINEPACKGEOM',
 'LINEPACKRATE',
 'MAINELEMENT',
 'MAV',
 'MI',
 'MK',
 'MKOND',
 'MMAX_INST',
 'MMIN_INST',
 'MVEC',
 'MVECMAX_INST',
 'MVECMIN_INST',
 'PAV',
 'PDAMPF',
 'PHR',
 'PHVEC',
 'PMAX',
 'PMIN',
 'PR',
 'PVEC',
 'PVECMAX_INST',
 'PVECMIN_INST',
 'QI2',
 'QK2',
 'QMAV',
 'QMI',
 'QMK',
 'QMMAX_INST',
 'QMMIN_INST',
 'QMVEC',
 'QSVB',
 'RHOAV',
 'RHOI',
 'RHOK',
 'RHOVEC',
 'SVEC',
 'TAV',
 'TI',
 'TK',
 'TTRVEC',
 'TVEC',
 'TVECMAX_INST',
 'TVECMIN_INST',
 'VAV',
 'VI',
 'VK',
 'VMAX_INST',
 'VMIN_INST',
 'VOLDA',
 'WALTERI',
 'WALTERK',
 'WVL',
 'ZAUS',
 'ZEIN',
 'ZHKNR',
 'ZVEC']

Let's also check what simulation timestamps are available in the model.

In [21]:
s3s.GetTimeStamps()[0] # list of simulation timestamps

['2023-02-13 00:00:00.000 +01:00',
 '2023-02-13 01:00:00.000 +01:00',
 '2023-02-13 02:00:00.000 +01:00',
 '2023-02-13 03:00:00.000 +01:00',
 '2023-02-13 04:00:00.000 +01:00',
 '2023-02-13 05:00:00.000 +01:00',
 '2023-02-13 06:00:00.000 +01:00',
 '2023-02-13 07:00:00.000 +01:00',
 '2023-02-13 08:00:00.000 +01:00',
 '2023-02-13 09:00:00.000 +01:00',
 '2023-02-13 10:00:00.000 +01:00',
 '2023-02-13 11:00:00.000 +01:00',
 '2023-02-13 12:00:00.000 +01:00',
 '2023-02-13 13:00:00.000 +01:00',
 '2023-02-13 14:00:00.000 +01:00',
 '2023-02-13 15:00:00.000 +01:00',
 '2023-02-13 16:00:00.000 +01:00',
 '2023-02-13 17:00:00.000 +01:00',
 '2023-02-13 18:00:00.000 +01:00',
 '2023-02-13 19:00:00.000 +01:00',
 '2023-02-13 20:00:00.000 +01:00',
 '2023-02-13 21:00:00.000 +01:00',
 '2023-02-13 22:00:00.000 +01:00',
 '2023-02-13 23:00:00.000 +01:00',
 '2023-02-14 00:00:00.000 +01:00']

In [22]:
s3s.GetTimeStamps()[1] # Static

'2023-02-13 00:00:00.000 +01:00'

In the below subcases differnt choices of parameters for creating the result dataframe manually are shown. Note that not all possible combination of paramters are shown here.

### Subcase 1: Static + All Result Values

In [23]:
(s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,properties=None # None => All available properties
                                        ,timestamps=[s3s.GetTimeStamps()[1]] # static
                                        )).head(3)

[2026-06-08 10:25:42,894] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe
[2026-06-08 10:25:43,055] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Only static timestamp 2023-02-13 00:00:00.000 +01:00 is used
[2026-06-08 10:25:43,056] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-06-08 10:25:43,063] INFO in sir3stoolkit.mantle.dataframes: [results] No properties given → using ALL result properties for ObjectTypes.Pipe.
[2026-06-08 10:25:43,065] INFO in sir3stoolkit.mantle.dataframes: [results] Using 82 result properties.
[2026-06-08 10:25:43,311] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values...
[2026-06-08 10:25:52,189] INFO in sir3stoolkit.mantle.dataframes: [results] 26724 fully NaN columns dropped.
[2026-06-08 10:25:54,722] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (1, 16244)


tk                                                                    5442010239090746007  \
name                                                                   Rohr V-K03S V-K13S   
end_nodes                      ('5669301360686511351', '5397948523091900401', '-1', '-1')   
property                                                                                A   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                                0.0           

tk                                                                           \
name                                                                          
end_nodes                                                                     
property                            DTTR    DWVERL DWVERLABS IAKTIV IRTRENN   
timestamp                                                                     
2023-02-13 00:00:00.000 +01:00  0.013437  35.86034  1.306369    0.0     0.0   

tk                                       \
name                                      
end_nodes                                 
property                             JV   
timestamp                                 
2023-02-13 00:00:00.000 +01:00  0.28607   

tk                                                                                \
name                                                                               
end_nodes                                                                          
property                                                                    MVEC   
timestamp                                                                          
2023-02-13 00:00:00.000 +01:00  14.68007\t14.68007\t14.68007\t14.68007\t14.68007   

tk                                                            \
name                                                           
end_nodes                                                      
property                          PDAMPF       PHR      PMIN   
timestamp                                                      
2023-02-13 00:00:00.000 +01:00  0.693796  0.010421  4.291257   

tk                                                                               \
name                                                                              
end_nodes                                                                         
property                                                                   PVEC   
timestamp                                                                         
2023-02-13 00:00:00.000 +01:00  4.502485\t4.449675\t4.39687\t4.344065\t4.291255   

tk                                                                               \
name                                                                              
end_nodes                                                                         
property                                                           PVECMAX_INST   
timestamp                                                                         
2023-02-13 00:00:00.000 +01:00  4.502485\t4.449675\t4.39687\t4.344065\t4.291255   

tk                                                                               \
name                                                                              
end_nodes                                                                         
property                                                           PVECMIN_INST   
timestamp                                                                         
2023-02-13 00:00:00.000 +01:00  4.502485\t4.449675\t4.39687\t4.344065\t4.291255   

tk                                                                      \
name                                                                     
end_nodes                                                                
property                            QMAV       QMI       QMK      RHOI   
timestamp               

Note that a multi index on the columns is created, holding the most important model_data.
- Level 0: tk (device ID)
- Level 1: name (device name)
- Level 2: end_nodes (tuple of connected node IDs as string)
- Level 3: property (result name)

As can be seen a lot of 99999.0 placeholder values are inserted, because we attempt to access many result values that SIR Calc can calculate but are not requested for this model. Therefore using all available result properties(properties=None), is generally not recommended.

### Subcase 2: All Timestamps + Specific Values

In [24]:
df_pipes_2 = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                                    ,properties=["DTTR", "PHR"] # Specify result properties of interest
                                                    ,timestamps=None # None => All available simulation timestamps
                                                    )

[2026-06-08 10:25:55,572] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe
[2026-06-08 10:25:55,574] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] No timestamps were given. Checking available simulation timestamps (SIR3S_Model.GetTimeStamps()[0]).
[2026-06-08 10:25:55,674] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] 25 simulation timestamp(s) are available.
[2026-06-08 10:25:55,772] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 25 timestamps: ['2023-02-13 00:00:00.000 +01:00', '2023-02-13 01:00:00.000 +01:00', '2023-02-13 02:00:00.000 +01:00', '2023-02-13 03:00:00.000 +01:00', '2023-02-13 04:00:00.000 +01:00', '2023-02-13 05:00:00.000 +01:00', '2023-02-13 06:00:00.000 +01:00', '2023-02-13 07:00:00.000 +01:00', '2023-02-13 08:00:00.000 +01:00', '2023-02-13 09:00:00.000 +01:00', '2023-02-13 10:00:00.000 +01:00', '2023-02-13 11:00:00.000 +01:00', '2023-02-13 12:00:00

In [25]:
df_pipes_2.head(3)

tk                                                                    5442010239090746007  \
name                                                                   Rohr V-K03S V-K13S   
end_nodes                      ('5669301360686511351', '5397948523091900401', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.013437           
2023-02-13 01:00:00.000 +01:00                                           0.013280           
2023-02-13 02:00:00.000 +01:00                                           0.013070           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.010421   
2023-02-13 01:00:00.000 +01:00  0.010659   
2023-02-13 02:00:00.000 +01:00  0.010989   

tk                                                                    4917786378639043296  \
name                                                                   Rohr V-K13S V-K23S   
end_nodes                      ('5397948523091900401', '5239335112004772156', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.025371           
2023-02-13 01:00:00.000 +01:00                                           0.025077           
2023-02-13 02:00:00.000 +01:00                                           0.024686           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.014625   
2023-02-13 01:00:00.000 +01:00  0.014954   
2023-02-13 02:00:00.000 +01:00  0.015408   

tk                                                                    4762482310382009633  \
name                                                                   Rohr V-K23S V-K33S   
end_nodes                      ('5239335112004772156', '5298886695042021307', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.017832           
2023-02-13 01:00:00.000 +01:00                                           0.017624           
2023-02-13 02:00:00.000 +01:00                                           0.017347           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.010872   
2023-02-13 01:00:00.000 +01:00  0.011119   
2023-02-13 02:00:00.000 +01:00  0.011460   

tk                                                                    4987229536643024523  \
name                                                                   Rohr V-K33S V-K43S   
end_nodes                      ('5298886695042021307', '4993257270457791438', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.002972           
2023-02-13 01:00:00.000 +01:00                                           0.002938           
2023-02-13 02:00:00.000 +01:00                         

We can access individual values as follows.

In [26]:
df_pipes_2.loc[
    "2023-02-13 00:00:00.000 +01:00",
    ("5442010239090746007", slice(None), slice(None), "DTTR")
].item()


0.01343678

### Subcase 3: Specific Timestamps + Specific Values

We can specify the list of timestamps to include either as a list of str timestamps ...

In [27]:
(s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,properties=["DTTR", "PHR"]
                                        ,timestamps=['2023-02-13 00:00:00.000 +01:00', '2023-02-13 05:00:00.000 +01:00', '2023-02-13 23:00:00.000 +01:00'] 
                                        )).head(3)

[2026-06-08 10:26:02,161] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe


[2026-06-08 10:26:02,377] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 3 timestamps: ['2023-02-13 00:00:00.000 +01:00', '2023-02-13 05:00:00.000 +01:00', '2023-02-13 23:00:00.000 +01:00']
[2026-06-08 10:26:02,381] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-06-08 10:26:02,384] INFO in sir3stoolkit.mantle.dataframes: [results] Using 2 result properties.
[2026-06-08 10:26:02,408] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values...
[2026-06-08 10:26:03,188] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.
[2026-06-08 10:26:03,410] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (3, 1048)


tk                                                                    5442010239090746007  \
name                                                                   Rohr V-K03S V-K13S   
end_nodes                      ('5669301360686511351', '5397948523091900401', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.013437           
2023-02-13 05:00:00.000 +01:00                                           0.008582           
2023-02-13 23:00:00.000 +01:00                                           0.012622           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.010421   
2023-02-13 05:00:00.000 +01:00  0.024731   
2023-02-13 23:00:00.000 +01:00  0.011750   

tk                                                                    4917786378639043296  \
name                                                                   Rohr V-K13S V-K23S   
end_nodes                      ('5397948523091900401', '5239335112004772156', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.025371           
2023-02-13 05:00:00.000 +01:00                                           0.016282           
2023-02-13 23:00:00.000 +01:00                                           0.023852           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.014625   
2023-02-13 05:00:00.000 +01:00  0.034240   
2023-02-13 23:00:00.000 +01:00  0.016454   

tk                                                                    4762482310382009633  \
name                                                                   Rohr V-K23S V-K33S   
end_nodes                      ('5239335112004772156', '5298886695042021307', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.017832           
2023-02-13 05:00:00.000 +01:00                                           0.011407           
2023-02-13 23:00:00.000 +01:00                                           0.016755           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.010872   
2023-02-13 05:00:00.000 +01:00  0.025619   
2023-02-13 23:00:00.000 +01:00  0.012245   

tk                                                                    4987229536643024523  \
name                                                                   Rohr V-K33S V-K43S   
end_nodes                      ('5298886695042021307', '4993257270457791438', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.002972           
2023-02-13 05:00:00.000 +01:00                                           0.001901           
2023-02-13 23:00:00.000 +01:00                         

... or as a list of int indices.

In [28]:
simulation_timestamps=s3s.GetTimeStamps()[0]

In [29]:
for idx, timestamp in enumerate(simulation_timestamps):
    print(idx, timestamp)

0 2023-02-13 00:00:00.000 +01:00
1 2023-02-13 01:00:00.000 +01:00
2 2023-02-13 02:00:00.000 +01:00
3 2023-02-13 03:00:00.000 +01:00
4 2023-02-13 04:00:00.000 +01:00
5 2023-02-13 05:00:00.000 +01:00
6 2023-02-13 06:00:00.000 +01:00
7 2023-02-13 07:00:00.000 +01:00
8 2023-02-13 08:00:00.000 +01:00
9 2023-02-13 09:00:00.000 +01:00
10 2023-02-13 10:00:00.000 +01:00
11 2023-02-13 11:00:00.000 +01:00
12 2023-02-13 12:00:00.000 +01:00
13 2023-02-13 13:00:00.000 +01:00
14 2023-02-13 14:00:00.000 +01:00
15 2023-02-13 15:00:00.000 +01:00
16 2023-02-13 16:00:00.000 +01:00
17 2023-02-13 17:00:00.000 +01:00
18 2023-02-13 18:00:00.000 +01:00
19 2023-02-13 19:00:00.000 +01:00
20 2023-02-13 20:00:00.000 +01:00
21 2023-02-13 21:00:00.000 +01:00
22 2023-02-13 22:00:00.000 +01:00
23 2023-02-13 23:00:00.000 +01:00
24 2023-02-14 00:00:00.000 +01:00


In [30]:
(s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,properties=["DTTR", "PHR"]
                                        ,timestamps=[0, 5, -2] 
                                        )).head(3)

[2026-06-08 10:26:04,713] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe
[2026-06-08 10:26:04,846] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 3 timestamps: ['2023-02-13 00:00:00.000 +01:00', '2023-02-13 05:00:00.000 +01:00', '2023-02-13 23:00:00.000 +01:00']
[2026-06-08 10:26:04,849] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-06-08 10:26:04,851] INFO in sir3stoolkit.mantle.dataframes: [results] Using 2 result properties.
[2026-06-08 10:26:04,895] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values...
[2026-06-08 10:26:05,527] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.
[2026-06-08 10:26:05,722] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (3, 1048)


tk                                                                    5442010239090746007  \
name                                                                   Rohr V-K03S V-K13S   
end_nodes                      ('5669301360686511351', '5397948523091900401', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.013437           
2023-02-13 05:00:00.000 +01:00                                           0.008582           
2023-02-13 23:00:00.000 +01:00                                           0.012622           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.010421   
2023-02-13 05:00:00.000 +01:00  0.024731   
2023-02-13 23:00:00.000 +01:00  0.011750   

tk                                                                    4917786378639043296  \
name                                                                   Rohr V-K13S V-K23S   
end_nodes                      ('5397948523091900401', '5239335112004772156', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.025371           
2023-02-13 05:00:00.000 +01:00                                           0.016282           
2023-02-13 23:00:00.000 +01:00                                           0.023852           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.014625   
2023-02-13 05:00:00.000 +01:00  0.034240   
2023-02-13 23:00:00.000 +01:00  0.016454   

tk                                                                    4762482310382009633  \
name                                                                   Rohr V-K23S V-K33S   
end_nodes                      ('5239335112004772156', '5298886695042021307', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.017832           
2023-02-13 05:00:00.000 +01:00                                           0.011407           
2023-02-13 23:00:00.000 +01:00                                           0.016755           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.010872   
2023-02-13 05:00:00.000 +01:00  0.025619   
2023-02-13 23:00:00.000 +01:00  0.012245   

tk                                                                    4987229536643024523  \
name                                                                   Rohr V-K33S V-K43S   
end_nodes                      ('5298886695042021307', '4993257270457791438', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.002972           
2023-02-13 05:00:00.000 +01:00                                           0.001901           
2023-02-13 23:00:00.000 +01:00                         

### Subcase 4: Specific Timestamps + Specific Values + Filtering

If we want to filter for more than just the container, we have to filter tks outside the function and then pass them. Works same way as for model_data.

In [31]:
all_tks = s3s.GetTksofElementType(s3s.ObjectTypes.Pipe)

In [32]:
len(all_tks)

524

In [33]:
good_tks = []

In [34]:
for tk in all_tks:
    if s3s.GetValue(tk, "FkCont")[0] == "5029128874972463118":
        if s3s.GetValue(tk, "bz.Irtrenn")[0] == "0": # Rohr nicht getrennt
            good_tks.append(tk)

In [35]:
len(good_tks)

480

In [36]:
(s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,tks=good_tks # here we specify that not all available tks should be used
                                        ,properties=["DTTR", "PHR"]
                                        ,timestamps=[0, 5, -2] 
                                        )).head(3)

[2026-06-08 10:26:07,091] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe
[2026-06-08 10:26:07,242] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 3 timestamps: ['2023-02-13 00:00:00.000 +01:00', '2023-02-13 05:00:00.000 +01:00', '2023-02-13 23:00:00.000 +01:00']
[2026-06-08 10:26:07,245] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-06-08 10:26:07,251] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] 480 tks remain after filtering for given tks.
[2026-06-08 10:26:07,256] INFO in sir3stoolkit.mantle.dataframes: [results] Using 2 result properties.
[2026-06-08 10:26:07,282] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values...
[2026-06-08 10:26:07,939] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.
[2026-06-08 10:26:08,134] INFO in sir3stoolkit.mantle.dataframes: [

tk                                                                    5442010239090746007  \
name                                                                   Rohr V-K03S V-K13S   
end_nodes                      ('5669301360686511351', '5397948523091900401', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.013437           
2023-02-13 05:00:00.000 +01:00                                           0.008582           
2023-02-13 23:00:00.000 +01:00                                           0.012622           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.010421   
2023-02-13 05:00:00.000 +01:00  0.024731   
2023-02-13 23:00:00.000 +01:00  0.011750   

tk                                                                    4917786378639043296  \
name                                                                   Rohr V-K13S V-K23S   
end_nodes                      ('5397948523091900401', '5239335112004772156', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.025371           
2023-02-13 05:00:00.000 +01:00                                           0.016282           
2023-02-13 23:00:00.000 +01:00                                           0.023852           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.014625   
2023-02-13 05:00:00.000 +01:00  0.034240   
2023-02-13 23:00:00.000 +01:00  0.016454   

tk                                                                    4762482310382009633  \
name                                                                   Rohr V-K23S V-K33S   
end_nodes                      ('5239335112004772156', '5298886695042021307', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.017832           
2023-02-13 05:00:00.000 +01:00                                           0.011407           
2023-02-13 23:00:00.000 +01:00                                           0.016755           

tk                                        \
name                                       
end_nodes                                  
property                             PHR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.010872   
2023-02-13 05:00:00.000 +01:00  0.025619   
2023-02-13 23:00:00.000 +01:00  0.012245   

tk                                                                    4987229536643024523  \
name                                                                   Rohr V-K33S V-K43S   
end_nodes                      ('5298886695042021307', '4993257270457791438', '-1', '-1')   
property                                                                             DTTR   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.002972           
2023-02-13 05:00:00.000 +01:00                                           0.001901           
2023-02-13 23:00:00.000 +01:00                         

### Subcase 5: Specific Vectorized results + Specific Timestamps

For pipes we have interior point result values. These are usually found as vectors with the postfix VEC.

In [37]:
available_result_props=s3s.GetResultProperties_from_elementType(s3s.ObjectTypes.Pipe,False)

In [38]:
available_result_props

['A',
 'ACALC',
 'CPI',
 'CPK',
 'DH',
 'DP',
 'DRAGRED',
 'DRAKONZ',
 'DSI',
 'DSK',
 'DTTR',
 'DWVERL',
 'DWVERLABS',
 'ETAAV',
 'FS',
 'HR',
 'HVEC',
 'IAKTIV',
 'IRTRENN',
 'JV',
 'JV2',
 'LAMBDA',
 'LECKEINAUS',
 'LECKMENGE',
 'LECKORT',
 'LINEPACK',
 'LINEPACKGEOM',
 'LINEPACKRATE',
 'MAINELEMENT',
 'MAV',
 'MI',
 'MK',
 'MKOND',
 'MMAX_INST',
 'MMIN_INST',
 'MVEC',
 'MVECMAX_INST',
 'MVECMIN_INST',
 'PAV',
 'PDAMPF',
 'PHR',
 'PHVEC',
 'PMAX',
 'PMIN',
 'PR',
 'PVEC',
 'PVECMAX_INST',
 'PVECMIN_INST',
 'QI2',
 'QK2',
 'QMAV',
 'QMI',
 'QMK',
 'QMMAX_INST',
 'QMMIN_INST',
 'QMVEC',
 'QSVB',
 'RHOAV',
 'RHOI',
 'RHOK',
 'RHOVEC',
 'SVEC',
 'TAV',
 'TI',
 'TK',
 'TTRVEC',
 'TVEC',
 'TVECMAX_INST',
 'TVECMIN_INST',
 'VAV',
 'VI',
 'VK',
 'VMAX_INST',
 'VMIN_INST',
 'VOLDA',
 'WALTERI',
 'WALTERK',
 'WVL',
 'ZAUS',
 'ZEIN',
 'ZHKNR',
 'ZVEC']

In [39]:
available_result_vector_props=[available_result_props for available_result_props in available_result_props if "VEC" in available_result_props]

In [40]:
available_result_vector_props

['HVEC',
 'MVEC',
 'MVECMAX_INST',
 'MVECMIN_INST',
 'PHVEC',
 'PVEC',
 'PVECMAX_INST',
 'PVECMIN_INST',
 'QMVEC',
 'RHOVEC',
 'SVEC',
 'TTRVEC',
 'TVEC',
 'TVECMAX_INST',
 'TVECMIN_INST',
 'ZVEC']

In [41]:
df_pipes = s3s.generate_element_results_dataframe(element_type=s3s.ObjectTypes.Pipe
                                        ,properties=["TTRVEC", "MVEC", "DTTR"]
                                        ,timestamps=[0, 5, -2] 
                                        )

[2026-06-08 10:26:09,351] INFO in sir3stoolkit.mantle.dataframes: [results] Generating results dataframe for element type: ObjectTypes.Pipe
[2026-06-08 10:26:09,528] INFO in sir3stoolkit.mantle.dataframes: [Resolving Timestamps] Using 3 timestamps: ['2023-02-13 00:00:00.000 +01:00', '2023-02-13 05:00:00.000 +01:00', '2023-02-13 23:00:00.000 +01:00']
[2026-06-08 10:26:09,528] INFO in sir3stoolkit.mantle.dataframes: [Resolving tks] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-06-08 10:26:09,546] INFO in sir3stoolkit.mantle.dataframes: [results] Using 3 result properties.
[2026-06-08 10:26:09,605] INFO in sir3stoolkit.mantle.dataframes: [results] Retrieving result values...
[2026-06-08 10:26:10,673] INFO in sir3stoolkit.mantle.dataframes: [results] 0 fully NaN columns dropped.
[2026-06-08 10:26:11,075] INFO in sir3stoolkit.mantle.dataframes: [results] Done. Shape: (3, 1572)


In [42]:
df_pipes.head(3)

tk                                                                    5442010239090746007  \
name                                                                   Rohr V-K03S V-K13S   
end_nodes                      ('5669301360686511351', '5397948523091900401', '-1', '-1')   
property                                                                           TTRVEC   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00  0.3227228\t0.326082\t0.3294412\t0.3328004\t0.3...           
2023-02-13 05:00:00.000 +01:00  0.2057882\t0.2079337\t0.2100792\t0.2122246\t0....           
2023-02-13 23:00:00.000 +01:00  0.3030591\t0.3062146\t0.3093701\t0.3125256\t0....           

tk                                                                                \
name                                                                               
end_nodes                                                                          
property                                                                    MVEC   
timestamp                                                                          
2023-02-13 00:00:00.000 +01:00  14.68007\t14.68007\t14.68007\t14.68007\t14.68007   
2023-02-13 05:00:00.000 +01:00  22.98332\t22.98332\t22.98332\t22.98332\t22.98332   
2023-02-13 23:00:00.000 +01:00  15.62753\t15.62753\t15.62753\t15.62753\t15.62753   

tk                                        \
name                                       
end_nodes                                  
property                            DTTR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.013437   
2023-02-13 05:00:00.000 +01:00  0.008582   
2023-02-13 23:00:00.000 +01:00  0.012622   

tk                                                                    4917786378639043296  \
name                                                                   Rohr V-K13S V-K23S   
end_nodes                      ('5397948523091900401', '5239335112004772156', '-1', '-1')   
property                                                                           TTRVEC   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00  0.3361596\t0.3403881\t0.3446167\t0.3488453\t0....           
2023-02-13 05:00:00.000 +01:00  0.2143701\t0.2170838\t0.2197974\t0.2225111\t0....           
2023-02-13 23:00:00.000 +01:00  0.3156811\t0.3196564\t0.3236316\t0.3276069\t0....           

tk                                                                                 \
name                                                                                
end_nodes                                                                           
property                                                                     MVEC   
timestamp                                                                           
2023-02-13 00:00:00.000 +01:00  8.24275\t8.24275\t8.24275\t8.24275\t8.24275\t8...   
2023-02-13 05:00:00.000 +01:00  12.84338\t12.84338\t12.84338\t12.84338\t12.843...   
2023-02-13 23:00:00.000 +01:00  8.767911\t8.767911\t8.767911\t8.767911\t8.7679...   

tk                                        \
name                                       
end_nodes                                  
property                            DTTR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.025371   
2023-02-13 05:00:00.000 +01:00  0.016282   
2023-02-13 23:00:00.000 +01:00  0.023852   

tk                                                                    4762482310382009633  \
name                                                                   Rohr V-K23S V-K33S   
end_nodes                      ('5239335112004772156', '5298886695042021307', '-1', '-1')   
property                                                                           TTRVEC   
timestamp                                                                     

As can be seen, the vectorized result values are written as strings with "/" seperators. For better access we can extend these values to a multiindex using [add_interior_points_as_multiindex()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.add_interior_points_as_multiindex).

In [43]:
df_pipes_interior_points = s3s.add_interior_points_to_start_end_sequence(df_pipes)

In [44]:
df_pipes_interior_points.head(3)

tk                                                                    5442010239090746007  \
name                                                                   Rohr V-K03S V-K13S   
end_nodes                      ('5669301360686511351', '5397948523091900401', '-1', '-1')   
property                                                                     TTRVEC_start   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.322723           
2023-02-13 05:00:00.000 +01:00                                           0.205788           
2023-02-13 23:00:00.000 +01:00                                           0.303059           

tk                                         \
name                                        
end_nodes                                   
property                       TTRVEC_end   
timestamp                                   
2023-02-13 00:00:00.000 +01:00   0.336160   
2023-02-13 05:00:00.000 +01:00   0.214370   
2023-02-13 23:00:00.000 +01:00   0.315681   

tk                                                                                 \
name                                                                                
end_nodes                                                                           
property                                                          TTRVEC_sequence   
timestamp                                                                           
2023-02-13 00:00:00.000 +01:00  (0.3227228, 0.326082, 0.3294412, 0.3328004, 0....   
2023-02-13 05:00:00.000 +01:00  (0.2057882, 0.2079337, 0.2100792, 0.2122246, 0...   
2023-02-13 23:00:00.000 +01:00  (0.3030591, 0.3062146, 0.3093701, 0.3125256, 0...   

tk                                                   \
name                                                  
end_nodes                                             
property                       MVEC_start  MVEC_end   
timestamp                                             
2023-02-13 00:00:00.000 +01:00   14.68007  14.68007   
2023-02-13 05:00:00.000 +01:00   22.98332  22.98332   
2023-02-13 23:00:00.000 +01:00   15.62753  15.62753   

tk                                                                                 \
name                                                                                
end_nodes                                                                           
property                                                            MVEC_sequence   
timestamp                                                                           
2023-02-13 00:00:00.000 +01:00  (14.68007, 14.68007, 14.68007, 14.68007, 14.68...   
2023-02-13 05:00:00.000 +01:00  (22.98332, 22.98332, 22.98332, 22.98332, 22.98...   
2023-02-13 23:00:00.000 +01:00  (15.62753, 15.62753, 15.62753, 15.62753, 15.62...   

tk                                        \
name                                       
end_nodes                                  
property                            DTTR   
timestamp                                  
2023-02-13 00:00:00.000 +01:00  0.013437   
2023-02-13 05:00:00.000 +01:00  0.008582   
2023-02-13 23:00:00.000 +01:00  0.012622   

tk                                                                    4917786378639043296  \
name                                                                   Rohr V-K13S V-K23S   
end_nodes                      ('5397948523091900401', '5239335112004772156', '-1', '-1')   
property                                                                     TTRVEC_start   
timestamp                                                                                   
2023-02-13 00:00:00.000 +01:00                                           0.336160           
2023-02-13 05:00:00.000 +01:00                                           0.214370           
2023-02-13 23:00:00.000 +01:00                                           0.315681      

We can access individual values as follows.

In [45]:
df_pipes_interior_points.loc[
    "2023-02-13 00:00:00.000 +01:00",
    ("5442010239090746007", slice(None), slice(None), "MVEC_sequence")
].item()


(14.68007, 14.68007, 14.68007, 14.68007, 14.68007)

We can also collapse the three time rows into one tuple.

In [46]:
df_pipes_simplified, timestamp_to_tuple_index = s3s.convert_rows_to_single_tuple_row(df_pipes_interior_points)

[2026-06-08 10:26:20,813] INFO in sir3stoolkit.mantle.dataframes: [rows_to_tuple_row] Collapsing dataframe rows into tuple row...
[2026-06-08 10:26:21,781] INFO in sir3stoolkit.mantle.dataframes: [rows_to_tuple_row] Done. Shape: (1, 3668)


In [47]:
df_pipes_simplified

tk                                               5442010239090746007  \
name                                              Rohr V-K03S V-K13S   
end_nodes ('5669301360686511351', '5397948523091900401', '-1', '-1')   
property                                                TTRVEC_start   
0                          (0.3227228, 0.2057882, 0.3030591)           

tk                                            \
name                                           
end_nodes                                      
property                          TTRVEC_end   
0          (0.3361596, 0.2143701, 0.3156811)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                     TTRVEC_sequence   
0          ((0.3227228, 0.326082, 0.3294412, 0.3328004, 0...   

tk                                                                         \
name                                                                        
end_nodes                                                                   
property                       MVEC_start                        MVEC_end   
0          (14.68007, 22.98332, 15.62753)  (14.68007, 22.98332, 15.62753)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                       MVEC_sequence   
0          ((14.68007, 14.68007, 14.68007, 14.68007, 14.6...   

tk                                                \
name                                               
end_nodes                                          
property                                    DTTR   
0          (0.01343678, 0.008581901, 0.01262201)   

tk                                               4917786378639043296  \
name                                              Rohr V-K13S V-K23S   
end_nodes ('5397948523091900401', '5239335112004772156', '-1', '-1')   
property                                                TTRVEC_start   
0                          (0.3361596, 0.2143701, 0.3156811)           

tk                                           \
name                                          
end_nodes                                     
property                         TTRVEC_end   
0          (0.361531, 0.2306521, 0.3395326)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                     TTRVEC_sequence   
0          ((0.3361596, 0.3403881, 0.3446167, 0.3488453, ...   

tk                                                                       \
name                                                                      
end_nodes                                                                 
property                      MVEC_start                       MVEC_end   
0          (8.24275, 12.84338, 8.767911)  (8.24275, 12.84338, 8.767911)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                       MVEC_sequence   
0          ((8.24275, 8.24275, 8.24275, 8.24275, 8.24275,...   

tk                                               \
name                                              
end_nodes                                         
property                                   DTTR   
0          (0.02537143, 0.01628198, 0.02385151)   

tk                                               4762482310382009633  \
name                                              Rohr V-K23S V-K33S   
end_nodes ('5239335112004772156', '5298886695042021307', '-1', '-1')   
property              

For every result value we now have a tuple with each entry corresponding to one simulation timestamp.

In [48]:
df_pipes_simplified.loc[
    0,
    ("5442010239090746007", slice(None), slice(None), "DTTR")
].item()

(0.01343678, 0.008581901, 0.01262201)

We can access individual values via the dict, that was returned.

In [49]:
df_pipes_simplified.loc[
    0,
    ("5442010239090746007", slice(None), slice(None), "DTTR")
].item()[timestamp_to_tuple_index['2023-02-13 00:00:00.000 +01:00']]

0.01343678

For vectorized result properties (pipe interior points) we will receive a tuple of tuples. With the time tuple as the higher level one.

In [50]:
df_pipes_simplified.loc[
    0,
    ("5442010239090746007", slice(None), slice(None), "MVEC_sequence")
].item()

((14.68007, 14.68007, 14.68007, 14.68007, 14.68007),
 (22.98332, 22.98332, 22.98332, 22.98332, 22.98332),
 (15.62753, 15.62753, 15.62753, 15.62753, 15.62753))

In [51]:
df_pipes_simplified.loc[
    0,
    ("5442010239090746007", slice(None), slice(None), "MVEC_sequence")
].item()[timestamp_to_tuple_index['2023-02-13 00:00:00.000 +01:00']]

(14.68007, 14.68007, 14.68007, 14.68007, 14.68007)

## Merge model_data and Result Dataframes (for one timestamp)

### Create Dataframes - For pipes

#### model_data

In [52]:
df_pipes_model_data = s3s.generate_element_model_data_dataframe(element_type=s3s.ObjectTypes.Pipe
                                                            ,properties=["L", "DN"]
                                                            ,geometry=True
                                                            )

[2026-06-08 10:26:23,584] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe
[2026-06-08 10:26:23,590] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-06-08 10:26:23,593] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 2 model_data properties.
[2026-06-08 10:26:23,593] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['L', 'DN'], geometry...
[2026-06-08 10:26:23,846] INFO in sir3stoolkit.mantle.dataframes: [model_data] Transforming DataFrame to GeoDataFrame successful with EPSG: 25832
[2026-06-08 10:26:23,846] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (524, 4)


In [53]:
df_pipes_model_data.head(3)

,tk,L,DN,geometry
0,5442010239090746007,36.42935,150,"LINESTRING (713620.268 5578828.419, 713602.295..."
1,4917786378639043296,57.21781,125,"LINESTRING (713602.295 5578860.106, 713574.062..."
2,4762482310382009633,40.99482,125,"LINESTRING (713574.062 5578909.873, 713553.84 ..."


#### Results

In [54]:
df_pipes_simplified.head(3)

tk                                               5442010239090746007  \
name                                              Rohr V-K03S V-K13S   
end_nodes ('5669301360686511351', '5397948523091900401', '-1', '-1')   
property                                                TTRVEC_start   
0                          (0.3227228, 0.2057882, 0.3030591)           

tk                                            \
name                                           
end_nodes                                      
property                          TTRVEC_end   
0          (0.3361596, 0.2143701, 0.3156811)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                     TTRVEC_sequence   
0          ((0.3227228, 0.326082, 0.3294412, 0.3328004, 0...   

tk                                                                         \
name                                                                        
end_nodes                                                                   
property                       MVEC_start                        MVEC_end   
0          (14.68007, 22.98332, 15.62753)  (14.68007, 22.98332, 15.62753)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                       MVEC_sequence   
0          ((14.68007, 14.68007, 14.68007, 14.68007, 14.6...   

tk                                                \
name                                               
end_nodes                                          
property                                    DTTR   
0          (0.01343678, 0.008581901, 0.01262201)   

tk                                               4917786378639043296  \
name                                              Rohr V-K13S V-K23S   
end_nodes ('5397948523091900401', '5239335112004772156', '-1', '-1')   
property                                                TTRVEC_start   
0                          (0.3361596, 0.2143701, 0.3156811)           

tk                                           \
name                                          
end_nodes                                     
property                         TTRVEC_end   
0          (0.361531, 0.2306521, 0.3395326)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                     TTRVEC_sequence   
0          ((0.3361596, 0.3403881, 0.3446167, 0.3488453, ...   

tk                                                                       \
name                                                                      
end_nodes                                                                 
property                      MVEC_start                       MVEC_end   
0          (8.24275, 12.84338, 8.767911)  (8.24275, 12.84338, 8.767911)   

tk                                                            \
name                                                           
end_nodes                                                      
property                                       MVEC_sequence   
0          ((8.24275, 8.24275, 8.24275, 8.24275, 8.24275,...   

tk                                               \
name                                              
end_nodes                                         
property                                   DTTR   
0          (0.02537143, 0.01628198, 0.02385151)   

tk                                               4762482310382009633  \
name                                              Rohr V-K23S V-K33S   
end_nodes ('5239335112004772156', '5298886695042021307', '-1', '-1')   
property              

### Merge

In [55]:
df_pipes_simplified.columns = df_pipes_simplified.columns.droplevel([1, 2])
df_pipes_simplified = df_pipes_simplified.T.unstack(level=0).T
df_pipes_simplified = df_pipes_simplified.droplevel(0, axis=0)
df_pipes = df_pipes_model_data.merge(on="tk",
                    how="outer",
                    right=df_pipes_simplified)

In [56]:
df_pipes.head(3)

,tk,L,DN,geometry,DTTR,MVEC_end,MVEC_sequence,MVEC_start,TTRVEC_end,TTRVEC_sequence,TTRVEC_start
0,4614463970292122863,7.780674,999,"LINESTRING (714262.483 5578857.42, 714269.543 ...","(7.780674, 7.780674, 7.780674)","(2.837623e-10, 1.142325e-09, 1.455192e-10)","((2.837623e-10, 2.837623e-10), (1.142325e-09, ...","(2.837623e-10, 1.142325e-09, 1.455192e-10)","(249.9251, 7.780674, 72269530.0)","((242.1444, 249.9251), (0.0, 7.780674), (72269...","(242.1444, 0.0, 72269520.0)"
1,4615723899944629797,64.287240,999,"LINESTRING (713738.297 5579219.902, 713793.23 ...","(64.28724, 64.28724, 64.28724)","(-2.983143e-10, 0.0, -1.964509e-10)","((-2.983143e-10, -2.983143e-10, -2.983143e-10,...","(-2.983143e-10, 0.0, -1.964509e-10)","(43.0797, 139.5106, 43.0797)","((107.3669, 98.18304, 88.99916, 79.81527, 70.6...","(107.3669, 139.5106, 107.3669)"
2,4621030304810285220,3.956838,100,"LINESTRING (713650.613 5578990.488, 713649.498...","(0.00229145, 0.001467105, 0.002153316)","(-4.251046, -6.639488, -4.523731)","((-4.251046, -4.251046), (-6.639488, -6.639488...","(-4.251046, -6.639488, -4.523731)","(0.05315937, 0.0340941, 0.04989032)","((0.05545082, 0.05315937), (0.0355612, 0.03409...","(0.05545082, 0.0355612, 0.05204364)"


To get this dataframe without having to perform all intermediate steps, use [generate_element_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_element_dataframe), described in Tutorial 52.